# RetailPulse — Exploratory Data Analysis

> **Data Notice:** This notebook analyses a SYNTHETIC dataset (seed=42, 10,000 rows, 2022–2024).  
> Findings are for portfolio demonstration only and do not represent real business performance.

## Notebook Outline
1. Load data
2. Schema inspection
3. Missing values and duplicates
4. Numeric distributions
5. Category analysis
6. Regional analysis
7. Time trends
8. Correlation analysis
9. Key findings

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import PROCESSED_FILE
from src.data_loader import load_processed, inspect_schema
from src.analytics import (
    kpi_summary, sales_by_category, monthly_sales_trend, 
    top_products_by_sales, sales_by_state
)

print('Imports OK')

## 1. Load Data

In [ ]:
df = load_processed()
schema = inspect_schema(df)

print(f'Shape: {df.shape}')
print(f'Date range: {schema["date_min"][:10]} → {schema["date_max"][:10]}')
df.head(5)

## 2. Schema Inspection

In [ ]:
print('Column dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

## 3. Headline KPIs

In [ ]:
kpis = kpi_summary(df)
for k, v in kpis.items():
    print(f'{k:30s}: {v}')

## 4. Category Analysis

In [ ]:
cat_df = sales_by_category(df)

fig = px.bar(
    cat_df,
    x='category',
    y='total_sales',
    color='profit_margin_pct',
    color_continuous_scale='Blues',
    title='Sales by Category (colour = profit margin %)',
    labels={'total_sales': 'Sales (INR)', 'category': 'Category'},
    text_auto=True,
)
fig.update_layout(plot_bgcolor='white', yaxis_tickformat='₹,.0f')
fig.show()

cat_df

## 5. Monthly Sales Trend

In [ ]:
monthly = monthly_sales_trend(df)

fig = px.line(
    monthly,
    x='year_month',
    y='total_sales',
    markers=True,
    title='Monthly Sales Trend (2022–2024)',
    labels={'year_month': 'Month', 'total_sales': 'Sales (INR)'},
)
fig.update_layout(plot_bgcolor='white', xaxis_tickangle=-45, yaxis_tickformat='₹,.0f')
fig.show()

monthly[['year_month', 'total_sales', 'mom_growth_pct']].tail(12)

## 6. Top Products

In [ ]:
top10 = top_products_by_sales(df, n=10)
top10[['product_name', 'category', 'total_sales', 'profit_margin_pct']]

## 7. Regional Analysis

In [ ]:
states = sales_by_state(df)

fig = px.bar(
    states,
    x='total_sales',
    y='state',
    orientation='h',
    color='profit_margin_pct',
    color_continuous_scale='Greens',
    title='Sales by State',
    labels={'total_sales': 'Sales (INR)', 'state': 'State'},
)
fig.update_layout(plot_bgcolor='white', xaxis_tickformat='₹,.0f',
                   yaxis={'categoryorder': 'total ascending'})
fig.show()

states

## 8. Numeric Distribution

In [ ]:
fig = px.histogram(
    df,
    x='sales_amount',
    nbins=50,
    title='Distribution of Sales Amount per Transaction Line',
    labels={'sales_amount': 'Sales Amount (INR)'},
    color_discrete_sequence=['#2E75B6'],
)
fig.update_layout(plot_bgcolor='white')
fig.show()

df['sales_amount'].describe()

## 9. Discount Distribution

In [ ]:
disc_counts = df['discount'].value_counts().reset_index()
disc_counts.columns = ['discount', 'count']
disc_counts = disc_counts.sort_values('discount')

fig = px.bar(
    disc_counts,
    x='discount',
    y='count',
    title='Transaction Count by Discount Level',
    labels={'discount': 'Discount Rate', 'count': 'Transactions'},
    color='count',
    color_continuous_scale='Blues',
)
fig.update_layout(plot_bgcolor='white', coloraxis_showscale=False)
fig.show()

## 10. Payment Method Analysis

In [ ]:
pay_counts = df.groupby('payment_method')['order_id'].nunique().reset_index()
pay_counts.columns = ['payment_method', 'orders']
pay_counts = pay_counts.sort_values('orders', ascending=False)

fig = px.pie(
    pay_counts,
    names='payment_method',
    values='orders',
    title='Order Share by Payment Method',
    hole=0.4,
)
fig.show()

pay_counts

## 11. Summary of Key Findings

> Remember: All findings are from SYNTHETIC data.

1. **Total sales:** ₹74.4M across 10,000 transactions
2. **AOV:** ₹7,444.72 — driven higher by Electronics orders
3. **Top category:** Electronics (20% of sales) but lowest margin (11%)
4. **Highest margin:** Beauty & Personal Care (~42%)
5. **Peak period:** Oct–Dec (festive season)
6. **Top state:** Maharashtra (~11.5% of sales)
7. **Average discount:** 11.4% (sales-weighted)
8. **Discount impact:** Margin drops from 29% at 0% discount to 22% at 31–40% discount